# ==================================================
# GRPO + RLVR MINI REFERENCE IMPLEMENTATION
# ==================================================

In [ ]:
#
# This notebook demonstrates:
#
# 1. RLVR Reward Verification
# 2. Group Relative Advantages (GRPO)
# 3. GRPO Loss Function
# 4. Policy Rollouts
# 5. Log Probability Computation
# 6. Training Step
# 7. Demo Example
#
# It is a simplified educational implementation.

from __future__ import annotations

# Regular expressions
import re

# Type hints
from typing import Iterable

# PyTorch
import torch

# ==================================================
# EXTRACT FINAL NUMBER
# ==================================================

In [ ]:
# Extracts the last number appearing in a response.
#
# Example:
#
# "2 + 3 = 5"
#     ↓
# "5"
#
# "Answer: 12.5"
#     ↓
# "12.5"

def extract_final_number(
    text: str
) -> str | None:

    """
    Return the last integer or decimal number
    found in a completion.
    """

    # Find all integers and decimals
    matches = re.findall(
        r"-?\d+(?:\.\d+)?",
        text
    )

    # Return last match
    return matches[-1] if matches else None

# ==================================================
# RLVR REWARD FUNCTION
# ==================================================

In [ ]:
# RLVR = Reinforcement Learning with Verifiable Rewards
#
# Reward:
#
# Correct Answer → 1.0
# Wrong Answer   → 0.0
#
# Example:
#
# Completion:
# "2 + 3 = 5"
#
# Gold:
# "5"
#
# Reward = 1

def verify_math_answer(
    completion: str,
    gold: str
) -> float:

    """
    Simple RLVR reward.
    """

    # Extract predicted answer
    predicted = extract_final_number(
        completion
    )

    # No answer found
    if predicted is None:

        return 0.0

    # Exact answer match
    return (
        1.0
        if predicted == str(gold)
        else 0.0
    )

# ==================================================
# GROUP RELATIVE ADVANTAGES
# ==================================================

In [ ]:
# GRPO normalizes rewards inside each group.
#
# Example:
#
# Rewards:
#
# [1,0,1,0]
#
# Mean = 0.5
#
# Advantages:
#
# [ +1
#   -1
#   +1
#   -1 ]
#
# This removes the need for a value model.

def group_advantages(
    rewards: torch.Tensor,
    eps: float = 1e-6
) -> torch.Tensor:

    """
    Normalize rewards inside each group.
    """

    # Group mean
    mean = rewards.mean(
        dim=1,
        keepdim=True
    )

    # Group standard deviation
    std = rewards.std(
        dim=1,
        keepdim=True
    ).clamp_min(eps)

    # Z-score normalization
    return (
        rewards - mean
    ) / std

# ==================================================
# GRPO LOSS
# ==================================================

In [ ]:
# Components:
#
# 1. PPO-style ratio
# 2. Clipping
# 3. KL penalty
#
# Final Objective:
#
# Surrogate Reward
#      -
# KL Penalty

def grpo_loss(
    logp: torch.Tensor,
    old_logp: torch.Tensor,
    ref_logp: torch.Tensor,
    advantages: torch.Tensor,
    eps: float = 0.2,
    beta: float = 0.04,
) -> torch.Tensor:

    """
    Compute GRPO loss.
    """

    # Policy ratio
    ratio = torch.exp(
        logp - old_logp
    )

    # PPO clipping
    clipped = ratio.clamp(
        1.0 - eps,
        1.0 + eps
    )

    # Expand advantages
    adv = advantages.unsqueeze(-1)

    # PPO surrogate objective
    surrogate = torch.minimum(
        ratio * adv,
        clipped * adv
    )

    # Reference-policy KL
    kl = logp - ref_logp

    # Convert maximization to minimization
    return -(
        surrogate -
        beta * kl
    ).mean()

# ==================================================
# SAMPLE COMPLETION GROUPS
# ==================================================

In [ ]:
# Generates multiple candidate answers
# for each prompt.
#
# Example:
#
# Prompt:
# "2+3"
#
# Generated:
#
# Answer 0
# Answer 1
# Answer 2
# Answer 3

def sample_group(
    policy,
    prompts: Iterable[str],
    group_size: int
):

    """
    Placeholder rollout function.
    """

    del policy

    completions = [

        [
            f"{prompt} answer {i}"

            for i in range(group_size)
        ]

        for prompt in prompts
    ]

    old_logp = torch.zeros(
        len(completions),
        group_size,
        4
    )

    return completions, old_logp

In [ ]:
# ==================================================
# TOKEN LOG PROBABILITIES
# ==================================================
#
# Scores generated text under a model.
#
# Used by:
#
# Policy Model
# Reference Model

def sequence_logprobs(
    model,
    prompts,
    completions
) -> torch.Tensor:

    """
    Placeholder implementation.
    """

    del model
    del prompts

    return torch.zeros(
        len(completions),
        len(completions[0]),
        4,
        requires_grad=True
    )

# ==================================================
# SINGLE GRPO TRAINING STEP
# ==================================================

In [ ]:
# Prompt
#   ↓
# Generate Completions
#   ↓
# Compute Rewards
#   ↓
# Compute Advantages
#   ↓
# Compute GRPO Loss
#   ↓
# Backpropagation
#   ↓
# Optimizer Update

def train_step(
    batch,
    policy,
    reference,
    optimizer,
    group_size: int = 4
):

    prompts, gold_answers = batch

    completions, old_logp = sample_group(
        policy,
        prompts,
        group_size
    )

    rewards = torch.tensor(

        [
            [
                verify_math_answer(
                    y,
                    gold
                )

                for y in group
            ]

            for group, gold
            in zip(
                completions,
                gold_answers
            )
        ],

        device=old_logp.device
    )

    advantages = group_advantages(
        rewards
    )

    logp = sequence_logprobs(
        policy,
        prompts,
        completions
    )

    with torch.no_grad():

        ref_logp = sequence_logprobs(
            reference,
            prompts,
            completions
        )

    loss = grpo_loss(
        logp,
        old_logp,
        ref_logp,
        advantages
    )

    optimizer.zero_grad(
        set_to_none=True
    )

    loss.backward()

    optimizer.step()

    return {

        "loss":
            float(loss.detach()),

        "reward":
            float(rewards.mean())
    }

# ==================================================
# DEMONSTRATION
# ==================================================

In [ ]:
def _demo():

    rewards = torch.tensor(
        [
            [1.0,0.0,1.0,0.0],
            [1.0,0.0,0.0,1.0]
        ]
    )

    advantages = group_advantages(
        rewards
    )

    print("rewards")
    print(rewards)

    print("advantages")
    print(advantages)

    logp = torch.randn(
        2,
        4,
        6
    ) * 0.05

    old_logp = torch.zeros_like(
        logp
    )

    ref_logp = torch.zeros_like(
        logp
    )

    loss = grpo_loss(
        logp,
        old_logp,
        ref_logp,
        advantages
    )

    print(
        f"demo loss: "
        f"{loss.item():.4f}"
    )

In [ ]:
if __name__ == "__main__":
    _demo()